# LSTM for Time Series

> Part of the **Machine Learning Learning Series**

## LSTM Forecasting
Deep learning approach that captures long-term dependencies in sequences.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
t = np.linspace(0, 8*np.pi, 500)
y = np.sin(t) + 0.5*np.sin(3*t) + np.random.randn(500)*0.1

seq_len = 30
X_seq, y_seq = [], []
for i in range(len(y) - seq_len):
    X_seq.append(y[i:i+seq_len])
    y_seq.append(y[i+seq_len])

X_seq = torch.FloatTensor(X_seq).unsqueeze(-1)
y_seq = torch.FloatTensor(y_seq).unsqueeze(-1)

train_size = int(0.8 * len(X_seq))
X_train, X_test = X_seq[:train_size], X_seq[train_size:]
y_train, y_test = y_seq[:train_size], y_seq[train_size:]

In [ ]:
class TSLSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(1, 64, 2, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(64, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

model = TSLSTMModel()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

for epoch in range(200):
    model.train()
    pred = model(X_train)
    loss = criterion(pred, y_train)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    
model.eval()
with torch.no_grad():
    test_pred = model(X_test).squeeze().numpy()
plt.figure(figsize=(12,4))
plt.plot(y_test.squeeze().numpy(), label='Actual', alpha=0.7)
plt.plot(test_pred, label='LSTM Forecast', alpha=0.7)
plt.legend()
plt.title('LSTM Time Series Forecast')
plt.show()